# Pair-form stopping

Autospec compares a tensor-product smooth and a joint tree for a screened
interaction. The adopted model must use the stopping iteration at which its
held-out risk justified adoption. This benchmark measures the correction of
[issue #535](https://github.com/fcela/mboostrs/issues/535) on three Gaussian
response generators, with five paired draws each. The first table includes
every non-mboostrs comparator from the original small-budget shootout.

In [ ]:
import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(ROOT))
from benches.report_pair_form_stopping import DIRECTORY, headline_table
from bench_tables import table

summary = json.loads((DIRECTORY / "summary.json").read_text())
display(headline_table(summary))

## Study design

The [frozen protocol](pair_form_stopping/PROTOCOL.md) uses the original
competitive shootout's disjoint training, validation, calibration and test
rows. Autospec receives Gaussian response semantics, the data and a budget of
32 fits, with one supplied validation fold. It receives no formula or planted
truth. Validation rows determine the specification and stopping iteration;
calibration rows determine a 90% split-conformal prediction interval; test rows
measure prediction error after those choices.

The before and after phases change only the library correction. Each runs in
fresh serial processes with one thread and a 240-second ceiling. All 30 runs
completed. The reporter verifies all planned draws and matching data and
partition hashes against every comparator. The competitors are the recorded
small-tier Ridge, EBM, CatBoost, LightGBM and XGBoost fits. Their settings and
search budgets differ from autospec's; these are pipeline comparisons.

Mean squared error (MSE) measures squared prediction error on the response
scale. The target is at most 5% excess MSE on every task relative to the best
complete comparator. Selecting that comparator using test error gives a
retrospective oracle comparison. These known generators are development
regressions; they do not constitute independent confirmation of the policy.

## Stopping evidence

The previous form picker retained each candidate's minimum cross-validated
risk and per-fold risks, but discarded the iteration at that minimum. After
adoption, the recommendation therefore fitted the winning formula at the
additive model's earlier stopping iteration. Its reported stopping evidence
also described the previous specification.

The correction retains the selected iteration, aggregate risk path and
full-data path already computed during form comparison. Adoption now updates
the formula, iteration, cached fit and stopping evidence together. The
adoption rule and number of CV fits remain unchanged. Statistical fitting and
CV continue through the existing Rust-backed model API.

On joint-boundary draws, mean test MSE falls from 0.6751 to 0.3943, a 41.6%
reduction. Excess over the best recorded comparator changes from +66.2% to
−2.9%. Four draws improve and one worsens by 0.9%. The paired values below
show that variation. Predictions on all ten smooth-additive and correlated-proxy
controls are byte-identical between phases, and every fit count is unchanged.

In [ ]:
paired = pl.DataFrame(summary["paired"])
display(table(
    paired.select("task", "seed", "before", "after", "change_pct", "before_mstop", "after_mstop"),
    stub="seed", stubhead="Draw", group="task",
    title="Stopping replay on the same outer observations",
    subtitle="Test MSE and relative change in percent; negative change improves prediction",
    labels={"before": "Before MSE", "after": "After MSE", "change_pct": "Change (%)",
            "before_mstop": "Before updates", "after_mstop": "After updates"},
    spanners={"Prediction": ["before", "after", "change_pct"],
              "Selected stopping": ["before_mstop", "after_mstop"]},
    formats={"change_pct": 1},
    notes=("One row per independent draw; phases share training, validation, calibration and test rows.",
           "Fit counts are unchanged on every row. This correction adds no candidate or CV sweep."),
))

## Scorecard profile

The assessment uses `mboostrs-scorecard/2.1`; both phases record its content
hash. Predictive validity (D11) scores 3: outer tests improve over the
prespecified Ridge reference on all three measured task families, with
spread and failures reported. Auditability (D10) scores 5 for this instrument's
recorded code, environment, loaded binary, input, partition and prediction
hashes. The profile below gives the evidence and anchor justification.

The remaining ten dimensions are unmeasured by this experiment. A selected
pair is not a validated discovery claim, and predictive interval coverage does
not measure uncertainty about an effect magnitude. The profile has no weighted
aggregate.

In [ ]:
card = json.loads((DIRECTORY / "scorecard.json").read_text())
profile = pl.DataFrame([
    {"dimension": f"{d['id']} {d['name']}", "status": d["status"],
     "score": d["score"], "justification": d["justification"] or "Outside this instrument's measurements."}
    for d in card["dimensions"]
])
display(table(
    profile, stub="dimension", stubhead="Scorecard dimension",
    title="Knowledge-discovery scorecard 2.1: measured evidence",
    subtitle="Scores 1–5, higher is better; unmeasured dimensions have no numeric score",
    spanners={"Assessment": ["status", "score", "justification"]},
    notes=("Scores describe this instrument and operating mode. No aggregate is defined for an incomplete profile.",),
))

## Strongest recorded search tier

A supplemental comparison also uses the best complete method across both
recorded search tiers. On joint-boundary draws the reference becomes tuned
CatBoost, with MSE 0.4057; corrected autospec is 2.8% lower. EBM's small tier
remains the strongest reference on the two additive controls. All three task
means therefore meet the 5% threshold against the strongest recorded tier as
well. This secondary analysis was added after the paired rerun and does not
replace the frozen small-tier comparison.


In [ ]:
strongest = pl.DataFrame(summary["supplemental_best_any_tier"])
display(table(
    strongest, stub="task", stubhead="Task",
    title="Supplement: strongest recorded competitor across both search tiers",
    subtitle="Test MSE and excess in percent; lower is better",
    formats={"excess_pct": 1},
    spanners={"Historical reference": ["best_method", "best_tier", "best_loss"],
              "After correction": ["after_loss", "excess_pct", "within_5pct"]},
    notes=("Secondary analysis added after the paired rerun; it does not replace the frozen small-tier comparison.",
           "The best complete method/tier is selected retrospectively from both small and tuned searches. Input and partition hashes are verified."),
))

## Remaining gaps

All three task means in this regression battery meet the 5% target. This
result does not establish the target on the real-data, causal, distributional,
uncertainty or workflow categories. The
[consolidated status](../../scorecard/status.md) records those broader gaps.

Several joint-boundary paths still select an iteration near the form picker's
cap. Replaying the measured iteration corrects the inconsistency; it does not
establish convergence. Candidate-path growth needs a separate paired study
that preserves the incumbent, budget accounting and additive controls. The
candidate-cap limitation is tracked in
[#536](https://github.com/fcela/mboostrs/issues/536).

## Reproduction and artifacts

The [before artifact](pair_form_stopping/before.json),
[after artifact](pair_form_stopping/after.json),
[paired summary](pair_form_stopping/summary.json) and
[scorecard profile](pair_form_stopping/scorecard.json) preserve the measured
results. The [standalone headline table](pair_form_stopping/headline.html)
contains the same comparisons as the opening table. Coverage and width remain
available per draw in the raw artifacts; this small experiment does not grade
uncertainty calibration.

From the repository root, rerun the after phase and regenerate the assessment:

```bash
.venv/bin/python -m benches.bench_pair_form_stopping --phase after
.venv/bin/python -m benches.report_pair_form_stopping
```

The before artifact was produced before applying the correction; preserve it
when reproducing the after phase. The raw files identify the loaded package
and binary separately from the source commit, including the dirty worktree
state at measurement.